## Some resources:
1. LLVM-tutor: https://github.com/banach-space/llvm-tutor/blob/main/lib/OpcodeCounter.cpp
2. LLVM source code: https://github.com/llvm/llvm-project/tree/main/llvm/include/llvm/IR
3. UFMG videos - https://www.youtube.com/@compilerslab/videos
4. https://www.cs.cmu.edu/afs/cs/academic/class/15745-s19/www/lectures/
5. https://www.cs.cornell.edu/courses/cs6120/2019fa/blog/loop-reduction/#strength-reduction
6. https://groups.seas.harvard.edu/courses/cs153/2018fa/lectures/Lec19-Loop-Optimization-II.pdf
7. https://www.cs.cornell.edu/courses/cs6120/2025fa/


## Some notes:
- A transformation pass takes IR, applies some modification, and returns which 
analyses remain valid.For example:
    - Replace a*b with a constant (constant folding)
    - Inline small functions
    - Reorder memory ops for efficiency
- Its key traits are:
    - Modifies IR.
    - Returns PreservedAnalyses to tell the pass manager what analyses are still valid.
    - Often uses results from analysis passes (e.g., LoopInfo, DominatorTree).

### Real world use:
- Rewrite IR to fit the custom dataflow ISA.
- Fold constants, tile loops, fuse ops, or lower MLIR dialects.
- Constant folding reduces compute and unlocks optimizations—direct energy savings on a battery-bound device.
- Memory stats (bytes, alignment, volatile/atomic, AS) quantify data movement, which usually dominates energy.
- Per-address-space breakdown maps neatly to dataflow fabrics and SRAM vs off-chip strategy.
- This pass is a building block: we can plug it before tiling/placement passes to guide tiling size, double buffering, and DMA scheduling.

## Typical workflow:

- Set up environment variables so that clang, opt, and llvm-config are found.
- Write the pass source code into a .cpp file.
- Compile the pass into a .dylib. (.dylib = your LLVM pass as a plugin, opt loads it at runtime and applies your transformation.)
- Create a test C file, lower it to LLVM IR (.ll).
- Run opt with your pass and capture the output.

### This work:
Constant Folding: If instruction (I) computes a result that can be known now (using LLVM’s DataLayout and standard math knowledge), replace it with the constant result.

- Constant folding removes runtime work, shrinks code, and unlocks downstream optimizations (dead code elimination, constant propagation, branch folding, etc.). For energy-efficient ML/IoT targets, folding reduces both cycles and instruction fetch energy
- We’re computing a lightweight memory access profile without running the program—perfect for early hardware-aware optimization: SRAM fit, tiling decisions, and DMA planning (all vital for an energy-focused dataflow ISA).
- Alignment affects both performance and legality of some vector/scatter ops. Tracking it quickly surfaces misaligned hot paths. This info is given by AlignHist

- How we implement:
    - For each instruction I, LLVM tries to evaluate it at compile time using ConstantFoldInstruction. It needs the DataLayout (DL) to know sizes/alignments/endianness. It uses TargetLibraryInfo (TLI) for known-libcall semantics (e.g., strlen("abc") → 3). If folding succeeds:
        - replaceAllUsesWith(C) rewires the IR so every user of I now uses the constant C. The old instruction I has no purpose anymore, so we push it into ToErase (don’t erase during the iteration or you’ll invalidate iterators). NumFolded increments for reporting.
        - we defer erasure--> erase after the walk to avoid iterator invalidation.
    - AlignHist:
        - Alignment insight: quick way to see how many accesses are 1/2/4/8/16-byte aligned—useful for vectorizer readiness and bus efficiency.
        - Per-AS stats: lets you spot, e.g., “too many DRAM loads,” or “unexpected MMIO usage”
        - Why DataLayout::getTypeStoreSize? It reports actual stored byte size for the value type—this is what matters for traffic/energy (not just element counts).
        - Why address space? It tells you which memory domain is accessed—super relevant if your hardware has different energy/latency per AS.
        - Volatile Flags: must not be removed/reordered; often MMIO → expensive.
        - Atomic Flags: has ordering/atomicity costs → performance/power implications.
        - We visit every instruction; classify LoadInst vs StoreInst; Alignment (LLVM 21): getAlign() returns Align, so .value() yields bytes.
      - If we folded anything, we conservatively say we didn’t preserve analyses (so downstream passes recompute safely). If nothing changed, we preserve all and return PreservedAnalyses::all()

In [ ]:
import sys, os

LLVM_HOME = "/opt/homebrew/Cellar/llvm/21.1.5"

os.environ["PATH"] = f"{LLVM_HOME}/bin:" + os.environ["PATH"]
os.environ["LDFLAGS"] = f"-L{LLVM_HOME}/lib"
os.environ["CPPFLAGS"] = f"-I{LLVM_HOME}/include"

!which clang
!which clang++
!which opt
!which llvm-config
!clang++ --version
!llvm-config --version

!echo 'export PATH="/opt/homebrew/opt/llvm/bin:$PATH"' >> ~/.zshrc
!echo 'export LDFLAGS="-L/opt/homebrew/opt/llvm/lib"' >> ~/.zshrc
!echo 'export CPPFLAGS="-I/opt/homebrew/opt/llvm/include"' >> ~/.zshrc
!source ~/.zshrc

In [ ]:
pass_code = r"""
#include "llvm/IR/PassManager.h"
#include "llvm/Passes/PassBuilder.h"
#include "llvm/Passes/PassPlugin.h"

#include "llvm/IR/Function.h"
#include "llvm/IR/Module.h"
#include "llvm/IR/Instructions.h"
#include "llvm/IR/InstrTypes.h"
#include "llvm/IR/CFG.h"
#include "llvm/IR/Constants.h"
#include "llvm/IR/DataLayout.h"

#include "llvm/Analysis/TargetLibraryInfo.h"
#include "llvm/Analysis/ConstantFolding.h"   // ConstantFoldInstruction

#include "llvm/Support/raw_ostream.h"

#include <map>
#include <string>
#include <vector>

using namespace llvm;

namespace {

struct ConstFoldMemStatsPass : public PassInfoMixin<ConstFoldMemStatsPass> {
  PreservedAnalyses run(Function &F, FunctionAnalysisManager &FAM) {
    errs() << "\n[constfold-memstats] Function: " << F.getName() << "\n";

    const DataLayout &DL = F.getParent()->getDataLayout();
    TargetLibraryInfo &TLI = FAM.getResult<TargetLibraryAnalysis>(F);

    // 1) Constant folding sweep 
    // We only do a simple, safe fold using ConstantFoldInstruction.
    // We don't try aggressive simplify that needs DT/AC, etc.
    unsigned NumFolded = 0;
    std::vector<Instruction*> ToErase;

    for (auto &BB : F) {
      for (auto &I : BB) {
        // Only fold if the instruction produces a value (non-void)
        if (I.getType()->isVoidTy()) continue;

        if (Constant *C = ConstantFoldInstruction(&I, DL, &TLI)) {
          // Replace all uses with folded constant and mark for erasure.
          I.replaceAllUsesWith(C);
          ToErase.push_back(&I);
          ++NumFolded;
        }
      }
    }

    for (Instruction *I : ToErase)
      I->eraseFromParent();

    errs() << "  Constant-folded instructions: " << NumFolded << "\n";

    // 2) Memory access statistics
    struct MemAgg {
      uint64_t Loads = 0, Stores = 0;
      uint64_t LoadBytes = 0, StoreBytes = 0;
      uint64_t Volatile = 0, Atomic = 0;
    };

    // Overall totals
    MemAgg Tot;

    // Per address-space aggregation (useful on GPUs/accelerators/custom ISAs)
    std::map<unsigned, MemAgg> ByAS;

    auto addLoad = [&](LoadInst *LI) {
      Type *VTy = LI->getType();
      uint64_t sz = DL.getTypeStoreSize(VTy);
      unsigned AS = LI->getPointerAddressSpace();

      ++Tot.Loads; Tot.LoadBytes += sz;
      auto &A = ByAS[AS];
      ++A.Loads; A.LoadBytes += sz;
      if (LI->isVolatile()) { ++Tot.Volatile; ++A.Volatile; }
      if (LI->isAtomic())   { ++Tot.Atomic;   ++A.Atomic;   }
    };

    auto addStore = [&](StoreInst *SI) {
      Type *VTy = SI->getValueOperand()->getType();
      uint64_t sz = DL.getTypeStoreSize(VTy);
      unsigned AS = SI->getPointerAddressSpace();

      ++Tot.Stores; Tot.StoreBytes += sz;
      auto &A = ByAS[AS];
      ++A.Stores; A.StoreBytes += sz;
      if (SI->isVolatile()) { ++Tot.Volatile; ++A.Volatile; }
      if (SI->isAtomic())   { ++Tot.Atomic;   ++A.Atomic;   }
    };

    // 3. alignment histogram (bytes)
    std::map<unsigned, uint64_t> AlignHist; // align -> count

    for (auto &BB : F) {
      for (auto &I : BB) {
        if (auto *LI = dyn_cast<LoadInst>(&I)) {
          addLoad(LI);
          unsigned A = LI->getAlign().value();
          AlignHist[A]++;
        } else if (auto *SI = dyn_cast<StoreInst>(&I)) {
          addStore(SI);
          unsigned A = SI->getAlign().value();
          AlignHist[A]++;
        }
      }
    }

    // Print summary
    errs() << "  Memory accesses (overall):\n";
    errs() << "    Loads:  " << Tot.Loads  << "  (" << Tot.LoadBytes  << " bytes)\n";
    errs() << "    Stores: " << Tot.Stores << "  (" << Tot.StoreBytes << " bytes)\n";
    errs() << "    Volatile ops: " << Tot.Volatile << "\n";
    errs() << "    Atomic ops:   " << Tot.Atomic   << "\n";

    if (!ByAS.empty()) {
      errs() << "  By address space:\n";
      for (auto &kv : ByAS) {
        unsigned AS = kv.first;
        const MemAgg &A = kv.second;
        errs() << "    AS" << AS
               << "  Loads="  << A.Loads  << " (" << A.LoadBytes  << "B)"
               << "  Stores=" << A.Stores << " (" << A.StoreBytes << "B)"
               << "  Vol="    << A.Volatile
               << "  Atom="   << A.Atomic << "\n";
      }
    }

    if (!AlignHist.empty()) {
      errs() << "  Alignment histogram (bytes -> count):\n";
      for (auto &kv : AlignHist) {
        errs() << "    " << kv.first << " -> " << kv.second << "\n";
      }
    }

    // We modified IR only by replacing folded instructions with constants.
    // That does not change analyses we don't query here, but be conservative:
    return NumFolded ? PreservedAnalyses::none() : PreservedAnalyses::all();
  }

  static bool isRequired() { return true; }
};

} // namespace

// -------- Plugin registration --------
extern "C" LLVM_ATTRIBUTE_WEAK PassPluginLibraryInfo llvmGetPassPluginInfo() {
  return {
    LLVM_PLUGIN_API_VERSION, "ConstFoldMemStats", "v0.1",
    [](PassBuilder &PB) {
      PB.registerPipelineParsingCallback(
        [](StringRef Name, FunctionPassManager &FPM,
           ArrayRef<PassBuilder::PipelineElement>) {
          if (Name == "constfold-memstats") {
            FPM.addPass(ConstFoldMemStatsPass());
            return true;
          }
          return false;
        });
    }
  };
}

"""

with open("ConstFoldMemStatsPass.cpp", "w") as f:
    f.write(pass_code)


In [ ]:
c_code = r"""
#include <stdio.h>

int global_array[4] = {1, 2, 3, 4};

int compute(int n) {
    // Constant expressions (foldable)
    int a = 2 * 3;            // 6
    int b = 10 + 20;          // 30
    int c = (4 - 1) * (2 + 1); // 9

    // Memory operations
    int tmp = global_array[0] + global_array[1];
    global_array[2] = tmp + c;

    // Constant conditional (foldable)
    if (4 > 2)
        return a + b + c;     // branch known true
    else
        return tmp;
}

int main() {
    int res = compute(10);
    printf("Result: %d\n", res);
    return 0;
}
"""

with open("fold.c", "w") as f:
    f.write(c_code)

In [ ]:
!/opt/homebrew/opt/llvm/bin/clang++ -std=c++20 -fPIC -shared \
  "$(/opt/homebrew/opt/llvm/bin/llvm-config --cxxflags --ldflags)" \
  -I/opt/homebrew/opt/llvm/include \
  -Wl,-undefined,dynamic_lookup \
  ConstFoldMemStatsPass.cpp -o ConstFoldMemStatsPass.dylib



In [ ]:
cat fold.ll

In [ ]:
!/opt/homebrew/opt/llvm/bin/clang -O0 -emit-llvm -S fold.c -o fold.ll
!/opt/homebrew/opt/llvm/bin/opt \
  -load-pass-plugin=./ConstFoldMemStatsPass.dylib \
  -passes="function(constfold-memstats)" -disable-output fold.ll